In [2]:
import io
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# =========================================================
# INITIALIZATION & SETUP
# =========================================================
# Download required NLTK data (uncomment if running for the first time)
# nltk.download('punkt')
# nltk.download('stopwords')

print("========== TEXT CLASSIFICATION SYSTEM ==========")

# 1. Load the Provided SMS Dataset

df = pd.read_csv("SMSSpamCollection", sep='\t', names=['label','text'])

# Map string labels to binary integers for classification metrics (spam=1, ham=0)
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})
print(f"Dataset loaded with {len(df)} records.\n")


# =========================================================
# 2. TEXT PREPROCESSING (Tokenization & Stopword Removal)
# =========================================================
print("========== PREPROCESSING TEXT ==========")
stop_words = set(stopwords.words('english'))
processed_texts = []

for text in df['text']:
    # Tokenization and lowercasing
    tokens = word_tokenize(str(text).lower())
    # Stopword removal and keeping alphanumeric tokens only
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    # Rejoin tokens into a cleaned string for the vectorizer
    processed_texts.append(" ".join(filtered_tokens))

df['processed_text'] = processed_texts
print("Sample of processed text:")
print(df['processed_text'].head(3).to_string(), "\n")


# =========================================================
# 3. FEATURE EXTRACTION (TF-IDF)
# =========================================================
print("========== TF-IDF FEATURE EXTRACTION ==========")
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_text'])
y = df['label_num']

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
print(f"Extracted {X_tfidf.shape[1]} numerical features.\n")


# =========================================================
# 4. MODEL TRAINING & SPLITTING
# =========================================================
# Splitting dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Initialize Models
nb_classifier = MultinomialNB()
lr_classifier = LogisticRegression(max_iter=1000)

# Train Models
nb_classifier.fit(X_train, y_train)
lr_classifier.fit(X_train, y_train)


# =========================================================
# 5. MODEL EVALUATION
# =========================================================
print("========== MODEL EVALUATION ==========")

# --- Naive Bayes Predictions and Metrics ---
nb_predictions = nb_classifier.predict(X_test)
nb_acc = accuracy_score(y_test, nb_predictions)
# Using zero_division parameter to prevent warnings on small datasets
nb_prec = precision_score(y_test, nb_predictions, zero_division=0) 
nb_rec = recall_score(y_test, nb_predictions, zero_division=0)
nb_f1 = f1_score(y_test, nb_predictions, zero_division=0)
nb_cm = confusion_matrix(y_test, nb_predictions)

print("--- Naïve Bayes Classifier ---")
print(f"Accuracy : {nb_acc:.4f}")
print(f"Precision: {nb_prec:.4f}")
print(f"Recall   : {nb_rec:.4f}")
print(f"F1-Score : {nb_f1:.4f}")
print(f"Confusion Matrix:\n{nb_cm}\n")

# --- Logistic Regression Predictions and Metrics ---
lr_predictions = lr_classifier.predict(X_test)
lr_acc = accuracy_score(y_test, lr_predictions)
lr_prec = precision_score(y_test, lr_predictions, zero_division=0)
lr_rec = recall_score(y_test, lr_predictions, zero_division=0)
lr_f1 = f1_score(y_test, lr_predictions, zero_division=0)
lr_cm = confusion_matrix(y_test, lr_predictions)

print("--- Logistic Regression Model ---")
print(f"Accuracy : {lr_acc:.4f}")
print(f"Precision: {lr_prec:.4f}")
print(f"Recall   : {lr_rec:.4f}")
print(f"F1-Score : {lr_f1:.4f}")
print(f"Confusion Matrix:\n{lr_cm}\n")


# =========================================================
# 6. PERFORMANCE COMPARISON & JUSTIFICATION
# =========================================================
print("========== RESULTS JUSTIFICATION ==========")
justification = """
* Note: Metrics may be skewed or uniform here due to the highly truncated sample size of the dataset provided in the script. On the full dataset, the following principles apply:

1. Naïve Bayes typically performs strongly as a baseline text classifier due to its assumption of feature independence, effectively capturing the probability of specific spam keywords (like 'FREE', 'WIN', 'URGENT') occurring.
2. Logistic Regression often achieves higher precision by learning complex decision boundaries and dynamically weighting TF-IDF features based on their true predictive power across the corpus, rather than strictly relying on joint probabilities.
3. If one model exhibits significantly higher Recall but lower Precision, it indicates the model is aggressively classifying messages as spam, causing false positives. A balanced F1-score highlights the most reliable overall model.
"""
print(justification)

========== TEXT CLASSIFICATION SYSTEM ==========
Dataset loaded with 5572 records.

========== PREPROCESSING TEXT ==========
Sample of processed text:
0    go jurong point crazy available bugis n great ...
1                              ok lar joking wif u oni
2    free entry 2 wkly comp win fa cup final tkts 2... 

========== TF-IDF FEATURE EXTRACTION ==========
TF-IDF Matrix Shape: (5572, 8079)
Extracted 8079 numerical features.

========== MODEL EVALUATION ==========
--- Naïve Bayes Classifier ---
Accuracy : 0.9731
Precision: 1.0000
Recall   : 0.7987
F1-Score : 0.8881
Confusion Matrix:
[[966   0]
 [ 30 119]]

--- Logistic Regression Model ---
Accuracy : 0.9543
Precision: 0.9712
Recall   : 0.6779
F1-Score : 0.7984
Confusion Matrix:
[[963   3]
 [ 48 101]]

========== RESULTS JUSTIFICATION ==========

* Note: Metrics may be skewed or uniform here due to the highly truncated sample size of the dataset provided in the script. On the full dataset, the following principles apply:

1. Naïve